# SFT 2단계 — QLoRA 학습

## 이 노트북이 하는 일
1단계에서 만든 `train_rft.jsonl`(정답 풀이 모음)로 모델을 학습시킵니다.

```
train_rft.jsonl  →  QLoRA 학습  →  LoRA 어댑터 (약 100MB)
```

## 사전 준비 ⚠️
`train_rft.jsonl`을 **Kaggle Dataset으로 업로드**해두셔야 합니다.
좌측 **Create → New Dataset** → 파일 업로드 → Private.
그다음 이 노트북 우측 **+ Add Input**에서 그 Dataset을 추가하세요.

## 실행 순서
1. Settings: **Accelerator = GPU T4 x2**, **Internet = On**
2. `[1]` `[2]` 실행 → **Run → Restart Session** → `[1]` 다시 → `[3]`부터

## vLLM을 안 쓰므로 protobuf 문제가 없습니다
이 노트북은 학습만 하고 추론은 안 합니다. 설치가 훨씬 단순해요.


---
## [1] 설정 ▶️ 항상 실행

### 🤔 QLoRA가 뭔가

**LoRA** — 30억 개 파라미터를 전부 학습하는 대신, 각 층에 **작은 행렬 두 개**를 덧붙여 그것만 학습합니다.
학습 대상이 30억 → 약 3천만 개로 줄어서 T4에서도 돌아갑니다.

**QLoRA** — 거기에 더해 **원본 모델을 4비트로 압축**해서 올립니다.
6.2GB → 약 2GB. 남은 메모리를 학습에 씁니다.

원본 가중치는 **얼지고(frozen)** LoRA 부분만 바뀝니다. 그래서 결과물이 100MB짜리 어댑터예요.

### 🤔 `EASY_KEEP`

RFT 데이터 7,695개 중 4,030개가 4/4(이미 잘 푸는) 문제 출신입니다.
전부 쓰면 학습의 절반이 **이미 잘하는 것**에 갑니다.

| EASY_KEEP | 총 샘플 | 흔들림 비중 |
|---|---|---|
| 4030 (전부) | 7,695 | 47.6% |
| **1800 (권장)** | **5,465** | **67.1%** |
| 0 | 3,665 | 100% |

0으로 두면 안 됩니다. 쉬운 풀이가 **출력 형식을 유지**하는 역할도 합니다.

### 🤔 하이퍼파라미터

- **`LORA_R=32`** — LoRA 행렬의 크기. 클수록 표현력↑ 메모리↑. 32가 일반적인 출발점
- **`LORA_ALPHA=64`** — LoRA의 영향력 배율. 보통 `r`의 2배
- **`LR=1e-4`** — LoRA는 학습 대상이 적어서 전체 파인튜닝(1e-5)보다 10배 큰 학습률을 씁니다
- **`EPOCHS=2`** — RFT는 있는 능력을 강화하는 것이라 많이 돌릴 필요가 없습니다. 3 이상은 과적합 위험

In [ ]:
# ── 데이터 ────────────────────────────────────────────
EASY_KEEP  = 1800      # 4/4 문제 출신 샘플 중 몇 개를 남길지
MAX_LEN    = 1024      # 토큰 길이 상한
EVAL_RATIO = 0.02      # 과적합 감시용 검증 비율

# ── LoRA ──────────────────────────────────────────────
LORA_R     = 32
LORA_ALPHA = 64
LORA_DROP  = 0.05

# ── 학습 ──────────────────────────────────────────────
EPOCHS     = 2
LR         = 1e-4
BATCH      = 1         # T4 메모리 한계
GRAD_ACC   = 16        # 실질 배치 = BATCH x GRAD_ACC = 16
SEED       = 42

MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"
OUT_DIR    = "/kaggle/working/qwen25-3b-rft-lora"
SYSTEM = ("You are an expert competition mathematician. Solve the problem step by step, "
          "concisely. The final answer is ALWAYS a single integer. "
          "End your response with the final integer inside \\boxed{}.")
# ──────────────────────────────────────────────────────
print(f"LoRA r={LORA_R} alpha={LORA_ALPHA} | lr={LR} | {EPOCHS}epoch | 실질배치={BATCH*GRAD_ACC}")

---
## [2] 설치 ⏭️ 세션을 끄지 않았으면 건너뛰기

| 패키지 | 역할 |
|---|---|
| `peft` | LoRA 구현 |
| `bitsandbytes` | 4비트 양자화 + 메모리 절약 옵티마이저 |
| `accelerate` | 학습 루프 보조 |

---
## ⛔ 설치 후 Run → Restart Session
재시작 후 **[1]부터** 다시 실행.

---

In [ ]:
!pip install -q -U peft bitsandbytes accelerate 2>&1 | tail -3

import torch, peft, bitsandbytes, transformers
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("GPU         :", torch.cuda.get_device_name(0))

---
## [3] 데이터 로드 + EASY_KEEP 적용 ▶️

`n_correct == 4`인 샘플을 `EASY_KEEP`개만 남기고, 나머지(1~3/4 출신)는 전부 씁니다.

**`random_state=SEED`로 고정**하므로 몇 번 실행해도 같은 샘플이 뽑힙니다.

In [ ]:
import json, glob, random
import pandas as pd

paths = glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
print("찾은 jsonl:", paths)
assert paths, "train_rft.jsonl 을 못 찾았습니다. 우측 + Add Input 으로 Dataset을 추가하세요."

records = [json.loads(l) for l in open(paths[0], encoding="utf-8")]
print(f"\n원본: {len(records):,}개")

easy = [r for r in records if r["n_correct"] == 4]
hard = [r for r in records if r["n_correct"] < 4]
print(f"  4/4 출신 : {len(easy):,}")
print(f"  1~3/4 출신: {len(hard):,}")

random.Random(SEED).shuffle(easy)
data = hard + easy[:EASY_KEEP]
random.Random(SEED).shuffle(data)

print(f"\n학습에 사용: {len(data):,}개 (흔들림 비중 {len(hard)/len(data):.1%})")

---
## [4] 토큰화 + 손실 마스킹 ▶️ (여기가 SFT의 핵심)

### 🤔 손실 마스킹이 뭔가

학습 데이터 한 건은 이렇게 생겼습니다.

```
<|im_start|>system
너는 수학 전문가다...<|im_end|>
<|im_start|>user
사과 17개를...<|im_end|>            ← 이 부분은 "입력"
<|im_start|>assistant
15개를 3으로 나누면... \boxed{5}<|im_end|>   ← 이 부분만 "정답"
```

모델이 배워야 할 건 **풀이를 쓰는 법**이지, 문제를 지어내는 법이 아닙니다.
그래서 프롬프트 구간의 라벨을 **`-100`**으로 채웁니다. PyTorch에서 `-100`은 "손실 계산에서 제외"를 뜻하는 약속된 값이에요.

이걸 안 하면 모델이 **문제 생성까지 학습**해서 성능이 떨어집니다.

### 🤔 왜 `apply_chat_template`을 두 번 부르나

- `add_generation_prompt=True` → 프롬프트만 (assistant 차례 직전까지)
- assistant 메시지 포함 → 전체

앞부분 길이만큼 마스킹하면 정확히 풀이 구간만 학습됩니다. 특수 토큰을 손으로 짜는 것보다 안전합니다.

In [ ]:
import torch
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print("eos:", repr(tok.eos_token), "| pad:", repr(tok.pad_token))

def build(r):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user",   "content": r["question"]}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full   = tok.apply_chat_template(
        msgs + [{"role": "assistant", "content": r["solution"]}], tokenize=False)

    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tok(full,   add_special_tokens=False)["input_ids"][:MAX_LEN]
    labels = list(f_ids)
    for i in range(min(len(p_ids), len(labels))):
        labels[i] = -100                      # 프롬프트 구간은 손실 제외
    return {"input_ids": f_ids, "labels": labels, "raw_len": len(tok(full)["input_ids"])}

ds = [build(r) for r in data]

lens = pd.Series([e["raw_len"] for e in ds])
print(f"\n토큰 길이  중앙값 {lens.median():.0f} / p95 {lens.quantile(.95):.0f} / 최대 {lens.max():.0f}")
trunc = (lens > MAX_LEN).mean()
print(f"MAX_LEN={MAX_LEN} 초과 비율: {trunc:.2%}")
if trunc > 0.05:
    print("⚠️ 5%를 넘습니다. MAX_LEN을 1536으로 올리는 걸 고려하세요 (메모리↑ 시간↑)")

n_eval = max(50, int(len(ds) * EVAL_RATIO))
eval_ds, train_ds = ds[:n_eval], ds[n_eval:]
print(f"\n학습 {len(train_ds):,} / 검증 {len(eval_ds):,}")
print(f"총 학습 토큰: {sum(len(e['input_ids']) for e in train_ds) * EPOCHS / 1e6:.1f}M")

---
## [5] 4비트 모델 + LoRA ▶️

### 옵션 하나씩

**`load_in_4bit`** — 원본 가중치를 4비트로 압축. 6.2GB → 약 2GB

**`bnb_4bit_quant_type="nf4"`** — NormalFloat4. 신경망 가중치의 분포(정규분포에 가까움)에 맞춰 설계된 4비트 형식이라, 일반 int4보다 정보 손실이 적습니다

**`bnb_4bit_compute_dtype=torch.float16`** — 계산은 fp16으로. **T4는 bf16 불가**라 여기서 bfloat16을 쓰면 안 됩니다

**`bnb_4bit_use_double_quant=True`** — 양자화 상수 자체를 한 번 더 압축. 메모리를 조금 더 아낍니다

**`target_modules`** — LoRA를 어느 층에 붙일지. 어텐션(q,k,v,o)과 MLP(gate,up,down) 전부에 붙이는 게 요즘 표준입니다

**`prepare_model_for_kbit_training`** — 4비트 모델을 학습 가능한 상태로 준비. LayerNorm을 fp32로 올리고, gradient checkpointing을 켭니다

### 🤔 gradient checkpointing

역전파에 필요한 중간값을 **저장하지 않고 필요할 때 다시 계산**합니다.
메모리를 크게 아끼는 대신 속도가 20~30% 느려집니다. T4 16GB에서는 필수예요.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4는 bf16 불가
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map={"": 0}, trust_remote_code=True)
model.config.use_cache = False               # 학습 중에는 KV cache 끄기
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROP,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

print(f"\nGPU 사용: {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## [6] 학습 ▶️ 3~5시간

### 옵션 설명

**`fp16=True`** — T4는 bf16 불가. fp16은 값 범위가 좁아 학습이 불안정할 수 있는데, Trainer가 **gradient scaling**으로 자동 보정합니다

**`optim="paged_adamw_8bit"`** — 옵티마이저 상태를 8비트로 저장. 일반 AdamW는 파라미터당 8바이트를 더 쓰는데, 이걸로 메모리를 크게 아낍니다

**`max_grad_norm=0.3`** — 기울기가 너무 커지면 잘라냅니다. QLoRA에서 흔히 쓰는 값

**`warmup_ratio=0.03`** — 처음 3%는 학습률을 0부터 서서히 올립니다. 초반 급격한 변화로 모델이 망가지는 걸 방지

**`lr_scheduler_type="cosine"`** — 학습률을 코사인 곡선으로 서서히 낮춥니다. 후반에 미세 조정

### 📊 진행 중 볼 것

- **train loss**가 꾸준히 내려가면 정상
- **eval loss**가 오르기 시작하면 **과적합**. 그 지점 전에서 멈춰야 합니다
- 진행바 **ETA가 6시간을 넘으면** 중단하고 `EASY_KEEP`이나 `EPOCHS`를 줄이세요

In [ ]:
from transformers import Trainer, TrainingArguments

def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    out = {"input_ids": [], "labels": [], "attention_mask": []}
    for b in batch:
        pad = m - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [tok.pad_token_id] * pad)
        out["labels"].append(b["labels"] + [-100] * pad)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * pad)
    return {k: torch.tensor(v) for k, v in out.items()}

args = TrainingArguments(
    output_dir="/kaggle/working/ckpt",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    fp16=True,                       # T4
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    per_device_eval_batch_size=1,
    save_strategy="no",              # 마지막에 수동 저장
    report_to="none",
    seed=SEED,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=eval_ds,
                  data_collator=collate)

trainer.train()

---
## [7] 어댑터 저장 ▶️

LoRA 어댑터만 저장합니다. 원본 6.2GB가 아니라 **약 100~200MB**예요.
원본 가중치는 안 바뀌었으니 저장할 필요가 없습니다.

### 저장 후 할 일
1. 아래 링크로 **zip 다운로드**
2. **Create → New Dataset**으로 업로드 (이름: `qwen25-3b-rft-lora`)
3. 추론 노트북에서 Input으로 추가해서 사용

### 추론 노트북에서 쓰는 법 (다음 단계)
```python
llm = LLM(model=MODEL_ID, ..., enable_lora=True, max_lora_rank=32)
from vllm.lora.request import LoRARequest
outs = llm.generate(prompts, sp,
                    lora_request=LoRARequest("rft", 1, "/kaggle/input/.../어댑터경로"))
```

In [ ]:
import shutil, os
model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)

for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f}  ({os.path.getsize(os.path.join(OUT_DIR,f))/1e6:.1f} MB)")

zip_path = shutil.make_archive("/kaggle/working/rft_lora", "zip", OUT_DIR)
print(f"\nzip: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")

from IPython.display import FileLink
FileLink("rft_lora.zip")